<a href="https://colab.research.google.com/github/dxda6216/ttron2excel/blob/main/ttron_data_file_to_excel_file_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from scipy import signal
from scipy.optimize import curve_fit
from google.colab import files

#@title Converting Taylortron TRACES file (TRACES.nnn) to Excel file
#@markdown **This script works only with a specific format of data files (*TRACES.nnn* files) generated by the [Taylortron](https://doi.org/10.1080/09291018209359765) in the Carl Johnson Lab.**

#@markdown 1. Input the experiment number (avoid spaces and special characters).
#@markdown 2. Input the experiment title (this field can be blank).
#@markdown 3. Input the date on which the experiment started.
#@markdown 4. Select 'Sinc Filter' ([firwin](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.firwin.html)) or '[Moving Average](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html)' for detrending.
#@markdown 5. If 'Sinc Filter' is chosen, adjust 'Sinc_Filter_Cutoff_Period_Hours' and 'Sinc_Filter_Order'.
#@markdown 6. If 'Moving Average' is chosen, adjust 'Window_size_for_trend_line_moving_average'.
#@markdown 7. **Runtime** -> **Restart and run all** (or press **Ctrl+M** and then press **Ctrl+F9**)
#@markdown 8. Wait until `Choose Files` or `Browse...` button appears below.
#@markdown 9. Click `Choose Files` or `Browse` button and select *TRACES.nnn* file in your computer.
#@markdown 10. Wait a while. three Excel files, one ZIP file, and one PDF file will be saved in "Downloads" folder in your computer.

#@markdown - The first Excel file will have multiple spreadsheets, conatining all the raw data, smoothed data, detranded data, and detected peak and trough times.
#@markdown - The second Excel file will have a single spreadsheet, conatining only the raw time series data. The interval time will be indicated as a sheet name of the Excel spreadsheet. This file can be opened with data analysis programs such as [pyBOAT](https://github.com/tensionhead/pyBOAT).
#@markdown - The third Excel file will have a single spreadsheet, conatining the results of damped sine fitting.
#@markdown - The ZIP file will contain separate data files (.dat files) for each of the channels. The .dat files can be opened with the [LumiCycle](https://actimetrics.com/products/lumicycle/) Analysis program.

def find_peaks_and_troughs(series, distance_in_hours=None, time_interval=None):
    """Find peak and trough indices in a (possibly NaN-containing) series.

    Returns the original-index labels of the detected peaks and troughs so
    they can be used with ``.loc`` on the source DataFrame.
    """
    if series.isnull().all():
        return [], []

    data_to_analyze = series.dropna().values

    if distance_in_hours and time_interval:
        distance_points = max(1, int(distance_in_hours / time_interval))
    else:
        distance_points = 5  # default distance in data points

    peaks, _ = signal.find_peaks(data_to_analyze, distance=distance_points)
    troughs, _ = signal.find_peaks(-data_to_analyze, distance=distance_points)

    original_indices = series.dropna().index
    return original_indices[peaks], original_indices[troughs]


def plot_channel_subplot(
    ax,
    x_scatter, y_scatter, scatter_label, scatter_color,
    x_line, y_line, line_label, line_color,
    x_lim_min, x_lim_max, y_lim_tuple,
    xticks_list, mit_on, mit_val,
    chlabel,
    current_plot_title=None,
):
    """Draw one channel's scatter + trend line onto ``ax``.

    Used by the grid-of-all-channels overview plots.
    """
    ax.scatter(x_scatter, y_scatter, s=0.1, c=scatter_color, label=scatter_label)
    ax.plot(x_line, y_line, line_color, linewidth=0.5, label=line_label)
    ax.set_xlim(x_lim_min, x_lim_max)
    if y_lim_tuple:
        ax.set_ylim(*y_lim_tuple)
    ax.set_xticks(xticks_list)
    if mit_on:
        ax.xaxis.set_minor_locator(MultipleLocator(mit_val))
    ax.grid(True, linewidth=0.5, color="lightgray", linestyle="--")
    ax.legend(loc="upper right", fontsize=4)
    if current_plot_title:
        ax.set_title(current_plot_title, fontsize=6)


def _actogram_double_plot_rows(hours, lod):
    """Turn absolute hours into (time_in_day, day_num, original_hour) rows.

    Each point is duplicated onto the previous day so the actogram double-plots.
    """
    rows = []
    for h in hours:
        day_num = h // lod
        time_in_day = h - day_num * lod
        rows.append((time_in_day, day_num, h))
        if day_num > 0:
            rows.append((time_in_day + lod, day_num - 1, h))
    return rows


def plot_individual_channel_summary(
    channelnumber, df, df_TL_MA, dtdf, dtdf_9PMA, Detrending_Method,
    Sinc_Filter_Cutoff_Period_Hours, filter_order,
    x_scale_min, x_scale_max, xtickslist, miton, mit,
    sub2label, sub3label, lod, last_time_point, time_interval,
):
    """Render the 3-panel (raw / detrended / actogram) summary figure for one channel.

    Relies on ``Experiment_number``, ``twss``, and ``pp`` being set as globals
    by the time this is called (as with the rest of this notebook's cells).
    """
    fig = plt.figure(figsize=(8.5, 11))
    fig.suptitle(Experiment_number + "   Ch # " + channelnumber, fontsize=12)

    # --- Subplot 1: raw data with trend line ---
    ax1 = plt.subplot(3, 1, 1)
    x = df["Hours"]
    y = df[channelnumber]
    x_cma = df_TL_MA["Hours"]
    y_cma = df_TL_MA[channelnumber]

    ax1.scatter(x, y, s=3.0, c="violet", label="Bioluminescence")
    if Detrending_Method == "Moving Average":
        ax1.plot(x_cma, y_cma, "-r", linewidth=1.0,
                  label=f"Trend line ({twss}-point moving average, centered)")
    else:
        ax1.plot(x_cma, y_cma, "-r", linewidth=1.0,
                  label=f"Trend line (Sinc Filter C={Sinc_Filter_Cutoff_Period_Hours}h O={filter_order})")
    ax1.set_xlim(x_scale_min, x_scale_max)
    y_scale_max = int(max(y) * 1.100)
    ax1.set_ylim(0, y_scale_max)
    ax1.set_xticks(xtickslist)
    if miton:
        ax1.xaxis.set_minor_locator(MultipleLocator(mit))
    ax1.set_xlabel("Hours", fontsize=10)
    ax1.set_ylabel("Bioluminescence", fontsize=10)
    ax1.grid(True, linewidth=0.5, color="lightgray", linestyle="--")
    ax1.legend(loc="upper right", fontsize=5)

    # --- Subplot 2: detrended data with smoothed line and peaks/troughs ---
    ax2 = plt.subplot(3, 1, 2)
    x_dt = dtdf["Hours"]
    y_dt = dtdf[channelnumber]
    x_dt_cma = dtdf_9PMA["Hours"]
    y_dt_cma = dtdf_9PMA[channelnumber]

    ax2.scatter(x_dt, y_dt, s=3.0, c="violet", label="Detrended bioluminescence")
    ax2.plot(x_dt_cma, y_dt_cma, "-b", linewidth=1.0,
              label="Smoothed line (9-point moving average, centered)")

    peaks_indices, troughs_indices = find_peaks_and_troughs(
        dtdf_9PMA[channelnumber], distance_in_hours=12, time_interval=time_interval)

    if len(peaks_indices) > 0:
        ax2.scatter(dtdf_9PMA.loc[peaks_indices, "Hours"], dtdf_9PMA.loc[peaks_indices, channelnumber],
                    marker="o", s=30, color="red", label="Peaks")
        if sub2label:
            for idx in peaks_indices:
                peak_time = dtdf_9PMA.loc[idx, "Hours"]
                peak_value = dtdf_9PMA.loc[idx, channelnumber]
                ax2.text(peak_time, peak_value + (ax2.get_ylim()[1] - ax2.get_ylim()[0]) * 0.05,
                         f"{peak_time:.2f} h", fontsize=5, color="red", ha="center", va="bottom")

    if len(troughs_indices) > 0:
        ax2.scatter(dtdf_9PMA.loc[troughs_indices, "Hours"], dtdf_9PMA.loc[troughs_indices, channelnumber],
                    marker="o", s=30, color="blue", label="Troughs")
        if sub2label:
            for idx in troughs_indices:
                trough_time = dtdf_9PMA.loc[idx, "Hours"]
                trough_value = dtdf_9PMA.loc[idx, channelnumber]
                ax2.text(trough_time, trough_value - (ax2.get_ylim()[1] - ax2.get_ylim()[0]) * 0.05,
                         f"{trough_time:.2f} h", fontsize=5, color="blue", ha="center", va="top")

    ax2.set_xlim(x_scale_min, x_scale_max)
    ax2.set_xticks(xtickslist)
    if miton:
        ax2.xaxis.set_minor_locator(MultipleLocator(mit))
    ax2.set_xlabel("Hours", fontsize=10)
    ax2.set_ylabel("Detrended bioluminescence", fontsize=10)
    ax2.grid(True, linewidth=0.5, color="lightgray", linestyle="--")
    ax2.legend(loc="upper right", fontsize=5)

    # --- Subplot 3: actogram of peaks and troughs ---
    ax3 = plt.subplot(3, 1, 3)

    peak_hours_actogram = dtdf_9PMA.loc[peaks_indices, "Hours"].values
    trough_hours_actogram = dtdf_9PMA.loc[troughs_indices, "Hours"].values

    peak_actogram_data = _actogram_double_plot_rows(peak_hours_actogram, lod)
    trough_actogram_data = _actogram_double_plot_rows(trough_hours_actogram, lod)

    if peak_actogram_data:
        peak_x, peak_y, peak_original_h = zip(*peak_actogram_data)
        ax3.scatter(peak_x, peak_y, marker="o", s=20, color="red", label="Peaks")
        if sub3label:
            for i in range(len(peak_x)):
                ax3.text(peak_x[i] + 0.5, peak_y[i], f"{peak_original_h[i]:.2f}", fontsize=5, color="red")

    if trough_actogram_data:
        trough_x, trough_y, trough_original_h = zip(*trough_actogram_data)
        ax3.scatter(trough_x, trough_y, marker="o", s=20, color="blue", label="Troughs")
        if sub3label:
            for i in range(len(trough_x)):
                ax3.text(trough_x[i] + 0.5, trough_y[i], f"{trough_original_h[i]:.2f}", fontsize=5, color="blue")

    if sub3label:
        ax3.set_xlim(-0.085 * lod, 2.085 * lod)
    else:
        ax3.set_xlim(0, lod * 2)  # actogram spans 0-48 hours for double plot

    ax3.set_xticks([0 * lod, 0.25 * lod, 0.5 * lod, 0.75 * lod, lod, 1.25 * lod, 1.5 * lod, 1.75 * lod, 2 * lod])

    if lod.is_integer():
        if lod % 4 == 0:
            ax3.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))
        elif lod % 2 == 0:
            ax3.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        else:
            ax3.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    else:
        if (lod * 10) % 4 == 0:
            ax3.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
        elif (lod * 10) % 2 == 0:
            ax3.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        else:
            ax3.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))

    if lod == 24:
        ax3.set_title("Peaks and Troughs", fontsize=10)
        ax3.set_ylabel("Days", fontsize=10)
    else:
        ax3.set_title(f"Peaks and Troughs (scaled x-axis: T = {lod} hours, T{lod})", fontsize=10)
        ax3.set_ylabel(f"Days (T{lod})", fontsize=10)

    ltpd = last_time_point // lod
    ax3.set_ylim(-ltpd * 0.05, ltpd * 1.05)
    ax3.yaxis.set_major_locator(MultipleLocator(base=1, offset=0))
    ax3.set_xlabel("Time (Hours)", fontsize=10)
    ax3.grid(True, linestyle="--", alpha=0.7)
    ax3.legend(loc="upper right", fontsize=6)
    ax3.invert_yaxis()  # Day 0 at the top

    plt.tight_layout(rect=[0, 0.02, 1, 0.98])
    pp.savefig(fig)
    plt.show()
    plt.close(fig)


def damped_sine_model(t, amplitude, period, phase, decay_rate, offset):
    """Damped sine model used to fit each channel's detrended rhythm."""
    return amplitude * np.exp(-decay_rate * t) * np.sin(2 * np.pi * t / period + phase) + offset

# ## Experiment parameters
#
# Fill in the fields below, then **Runtime → Restart and run all**.

#@markdown **Experiment identification**
Experiment_number = 'CYxxx' #@param {type:"string"}
Experiment_title = '' #@param {type:"string"}
Date_experiment_started = '2026-01-01' #@param {type:"date"}

#@markdown **Detrending method**
Detrending_Method = "Sinc Filter" #@param ["Sinc Filter", "Moving Average"]

#@markdown **Detrending parameters** &mdash; only the block matching the method above is used.

# Sinc Filter parameters (firwin: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.firwin.html)
Sinc_Filter_Cutoff_Period_Hours = 48  # @param {type:"slider", min:1, max:240, step:1}
Sinc_Filter_Order = 101  # @param {type:"slider", min:1, max:361, step:2}

# Moving-average parameters (rolling mean: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html)
Window_size_for_trend_line_moving_average = 24  # @param {type:"slider", min:1, max:120, step:1}

#@markdown **Overall raw/detrended data plots**
Data_Plotting = "Plotting the channel 00 data last" #@param ["Plotting the channel 00 data first", "Plotting the channel 00 data last"]

if Data_Plotting == "Plotting the channel 00 data first":
    chlist = list(range(30))
else:
    chlist = list(range(1, 30)) + [0]

Subplot_Matrix = "3-column by 10-row" #@param ["3-column by 10-row", "4-column by 8-row"]

if Subplot_Matrix == "3-column by 10-row":
    subp_raw, subp_col = 10, 3
else:
    subp_raw, subp_col = 8, 4

#@markdown **X-axis major and minor ticks**
Major_Ticks = "Every 24 hours" #@param ["Every 12 hours", "Every 24 hours", "Every 48 hours"]
major_ticks_map = {"Every 12 hours": 12, "Every 24 hours": 24, "Every 48 hours": 48}
mjt = major_ticks_map[Major_Ticks]

Minor_Ticks = "Every 12 hours" #@param ["No minor ticks", "Every 2 hours", "Every 4 hours", "Every 6 hours", "Every 12 hours", "Every 24 hours"]
minor_ticks_map = {
    "No minor ticks": 0, "Every 2 hours": 2, "Every 4 hours": 4,
    "Every 6 hours": 6, "Every 12 hours": 12, "Every 24 hours": 24,
}
mit = minor_ticks_map[Minor_Ticks]
miton = mit != 0 and mit < mjt

#@markdown **Label peaks and troughs in the detrended data plot?**
Label_peaks_and_troughs_in_detrended_data_plot = "Yes" #@param ["Yes", "No"]
sub2label = Label_peaks_and_troughs_in_detrended_data_plot == "Yes"

#@markdown **Double-plot actogram**
Actogram_X_axis_scale = 24  # @param {type:"slider", min:12, max:60, step:0.1}
lod = Actogram_X_axis_scale

Label_peaks_and_troughs_in_actogram = "Yes" #@param ["Yes", "No"]
sub3label = Label_peaks_and_troughs_in_actogram == "Yes"

#@markdown **Time range (in hours) for fitting the damped sine curve**
Start_Hour_for_Fitting = 24  # @param {type:"slider", min:0.0, max:360.0, step:1}
End_Hour_for_Fitting = 120  # @param {type:"slider", min:0.0, max:360.0, step:1}

if End_Hour_for_Fitting - Start_Hour_for_Fitting <= 23:
    Start_Hour_for_Fitting, End_Hour_for_Fitting = 24, 120

# ## Upload and load the TRACES file

### Remove leftover output files from a previous run
!rm -rf *.xlsx *.pdf *.dat *.zip TRACES.* Traces.* traces.*

### Upload the TRACES.xxx file
uploaded = files.upload()
ttronfilename = next(iter(uploaded))

sttime = datetime.now(timezone.utc)
print("\nStarted at " + sttime.strftime("%Y-%m-%d %H:%M:%S") + " (UTC)")

### Read the uploaded TRACES file into a DataFrame
print("\nReading the data...")

colnames = ["Hours"] + [str(k).zfill(2) for k in range(30)]

df = pd.read_csv(
    ttronfilename, header=None, sep="\t", skiprows=3, skipfooter=1,
    index_col=False, names=colnames, engine="python",
)
df2 = df.iloc[:, 0:31]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
print("\nRaw Data:")
display(df)

number_of_rows = len(df.index)
last_row_index = number_of_rows - 1
first_time_point = df.loc[0, "Hours"]
last_time_point = df.loc[last_row_index, "Hours"]
total_time = last_time_point - first_time_point
total_time_in_days = total_time / 24
time_interval = total_time / last_row_index
excel2_sheet_name = f"INTVL = {time_interval:.15f} h"

print(f"Number of rows: {number_of_rows}")
print(f"First time point: {first_time_point} h")
print(f"Last time point: {last_time_point} h")
print(f"Total time duration: {total_time} h = {total_time_in_days} days")
print(f"Average time interval: {time_interval} h")

# ## Compute the trend line and detrended data

print("\nCalculating trend line and detrended data using " + Detrending_Method + "...")

df_5PMA = df.rolling(window=5, center=True, min_periods=1).mean()
df_9PMA = df.rolling(window=9, center=True, min_periods=1).mean()

if Detrending_Method == "Moving Average":
    tws = math.ceil(Window_size_for_trend_line_moving_average / time_interval)
    if tws % 2 == 0:
        tws += 1
    twss = int(tws)
    twst = time_interval * (twss - 1)
    print(f"Window size for trend line (Moving Average): {twst:.6f} h ({twss} points)")

    df_TL_MA = df.rolling(window=twss, center=True, min_periods=1).mean()
    df_TL_MA = df_TL_MA.bfill().ffill()  # fill in the boundary NaNs
    trendline_sheet_name = f"Trend line ({twss}PMA)"
    filter_order = None  # not used for this method, but referenced later for plot labels

else:  # Sinc Filter
    sampling_rate = 1 / time_interval  # samples per hour
    cutoff_frequency = 1 / Sinc_Filter_Cutoff_Period_Hours  # cycles per hour
    nyquist_frequency = 0.5 * sampling_rate
    norm_cutoff = cutoff_frequency / nyquist_frequency
    filter_order = Sinc_Filter_Order

    # Keep the filter order odd and small enough for filtfilt to run reliably
    max_filter_order_for_data = int(number_of_rows / 3) - 1
    if filter_order >= max_filter_order_for_data:
        print(f"Warning: Sinc Filter Order ({filter_order}) is too high for data length ({number_of_rows} points).")
        filter_order = max_filter_order_for_data
        if filter_order % 2 == 0:
            filter_order -= 1
        filter_order = max(filter_order, 1)
        print(f"Adjusting Sinc Filter Order to {filter_order} for stability with filtfilt.")
    if filter_order % 2 == 0:
        filter_order += 1

    print(f"Sampling Rate: {sampling_rate:.2f} samples/hour")
    print(f"Cutoff Period for Sinc Filter: {Sinc_Filter_Cutoff_Period_Hours} hours")
    print(f"Cutoff Frequency: {cutoff_frequency:.4f} cycles/hour")
    print(f"Normalized Cutoff Frequency: {norm_cutoff:.4f}")
    print(f"Sinc Filter Order: {filter_order}")

    sinc_filter_coeffs = signal.firwin(filter_order, norm_cutoff, pass_zero="lowpass")

    df_TL_MA = df.copy()
    for k in range(30):
        channelnumber = str(k).zfill(2)
        if df[channelnumber].isnull().all():
            df_TL_MA[channelnumber] = np.nan
            continue

        # Interpolate NaNs before filtering (better than a mean fill)
        data_to_filter = df[channelnumber].interpolate(method="linear", limit_direction="both")

        try:
            df_TL_MA[channelnumber] = signal.filtfilt(sinc_filter_coeffs, [1.0], data_to_filter)
        except ValueError as e:
            print(f"Warning: Could not apply sinc filter to channel {channelnumber}. Error: {e}")
            print("Consider reducing filter_order or increasing data length.")
            df_TL_MA[channelnumber] = np.nan

    trendline_sheet_name = f"Trend line (Sinc Filter C={Sinc_Filter_Cutoff_Period_Hours}h O={filter_order})"

# Detrend by subtracting the trend line (keep the "Hours" column intact)
dtdf = df - df_TL_MA
dtdf["Hours"] = df_TL_MA["Hours"]
dtdf_5PMA = dtdf.rolling(window=5, center=True, min_periods=1).mean()
dtdf_9PMA = dtdf.rolling(window=9, center=True, min_periods=1).mean()

# ## Write the main Excel output file

print("\nGenerating an Excel file...")

now_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
trend_line_parameter = (
    f"{Window_size_for_trend_line_moving_average}h window"
    if Detrending_Method == "Moving Average"
    else f"{Sinc_Filter_Cutoff_Period_Hours}h cutoff, {Sinc_Filter_Order} order"
)

note_df = pd.DataFrame.from_dict({
    "A": ["Experiment Number", "Experiment Title", "Experiment Start Date", "TRACES File", "",
          "Number of Time Points", "Total Time Duration (Hours)", "Average Time Interval (Hours)",
          "Detrending Method", "Trend Line Parameter", "", "Data Processed Date and Time (UTC)"],
    "B": [""] * 12,
    "C": [""] * 12,
    "D": [""] * 12,
    "E": [Experiment_number, Experiment_title, Date_experiment_started, ttronfilename, "",
          number_of_rows, total_time, time_interval, Detrending_Method, trend_line_parameter, "", now_str],
})

outputexcelfilename = f"{Experiment_number}_data.xlsx"
with pd.ExcelWriter(outputexcelfilename) as writer:
    note_df.to_excel(writer, sheet_name="Note", index=None, header=False)
    df.to_excel(writer, sheet_name="Raw Data")
    df_5PMA.to_excel(writer, sheet_name="5-point moving average (5PMA)")
    df_9PMA.to_excel(writer, sheet_name="9-point moving average (9PMA)")
    df_TL_MA.to_excel(writer, sheet_name=trendline_sheet_name)
    dtdf.to_excel(writer, sheet_name="Detrended Data")
    dtdf_5PMA.to_excel(writer, sheet_name="Detrended Data 5PMA")
    dtdf_9PMA.to_excel(writer, sheet_name="Detrended Data 9PMA")

    all_peaks_troughs_data = []
    for k in range(30):
        channelnumber = str(k).zfill(2)

        dfx = pd.DataFrame({
            "Hours": df["Hours"],
            "Raw_data": df[channelnumber],
            "5PMA": df_5PMA[channelnumber],
            "9PMA": df_9PMA[channelnumber],
            "trend_line": df_TL_MA[channelnumber],
            "detrended_data": dtdf[channelnumber],
            "detrended_data_5PMA": dtdf_5PMA[channelnumber],
            "detrended_data_9PMA": dtdf_9PMA[channelnumber],
        })
        dfx.to_excel(writer, sheet_name=f"Channel {channelnumber}")

        peaks_indices, troughs_indices = find_peaks_and_troughs(
            dtdf_9PMA[channelnumber], distance_in_hours=12, time_interval=time_interval)
        for peak_index in peaks_indices:
            all_peaks_troughs_data.append({
                "Channel": channelnumber, "Type": "Peak",
                "Time (Hours)": dtdf_9PMA.loc[peak_index, "Hours"],
            })
        for trough_index in troughs_indices:
            all_peaks_troughs_data.append({
                "Channel": channelnumber, "Type": "Trough",
                "Time (Hours)": dtdf_9PMA.loc[trough_index, "Hours"],
            })

    if all_peaks_troughs_data:
        pd.DataFrame(all_peaks_troughs_data).to_excel(writer, sheet_name="Peaks and Troughs", index=False)
    else:
        print("No peaks or troughs detected for any channel.")

print(f"\nExcel file: {outputexcelfilename} has been generated.")

# ## Write the raw-data-only Excel file (for pyBOAT, etc.)

outputexcelfilename2 = f"{Experiment_number}_data_2.xlsx"
with pd.ExcelWriter(outputexcelfilename2) as writer:
    df2.to_excel(writer, sheet_name=excel2_sheet_name, index=None, header=True)

print(f"\nExcel file: {outputexcelfilename2} has been generated.")

# ## Export per-channel .dat files (for LumiCycle Analysis)

print("\nGenerating a data file for each channel (.dat files)...")
df["Days"] = df["Hours"] / 24.0
for k in range(30):
    channelnumber = str(k).zfill(2)
    df.to_csv(f"{channelnumber}.dat", header=False, index=False, sep="\t", columns=["Days", channelnumber])

print("\nPacking .dat files into a zip file...")
zip_output_filename = f"{Experiment_number}_data.zip"
!zip -r {zip_output_filename} ./*.dat

# ## Plot overview figures (grid of all channels)

print("\nPlotting...")
plot_output_pdf = f"{Experiment_number}_data_plots.pdf"

x_hours = df["Hours"]
x_scale_min = int(math.floor(min(x_hours) / 24)) * 24
x_scale_max = int(math.ceil(max(x_hours) / 12)) * 12 + 12
xtickslist = list(range(x_scale_min, x_scale_max, mjt))

pp = PdfPages(plot_output_pdf)
plt.rcParams.update({"figure.max_open_warning": 0})


def _set_overview_plot_style():
    """rcParams shared by the two grid-of-channels overview figures."""
    plt.rc("font", size=5)
    plt.rc("axes", titlesize=4)
    plt.rc("axes", labelsize=4)
    plt.rc("xtick", labelsize=4)
    plt.rc("ytick", labelsize=4)
    plt.rc("legend", fontsize=3)
    plt.rc("figure", titlesize=5)


# --- Overall raw data with trend line ---
_set_overview_plot_style()
fig = plt.figure(figsize=(11, 8.5))
fig.subplots_adjust(hspace=0.15)
fig.suptitle(Experiment_number, fontsize=12)

for subplotnumber, k in enumerate(chlist, start=1):
    channelnumber = str(k).zfill(2)
    ax = plt.subplot(subp_raw, subp_col, subplotnumber)
    y = df[channelnumber]
    chlabel = "Ch # " + channelnumber
    y_scale_max = int(max(y) * 1.100)

    plot_channel_subplot(
        ax=ax,
        x_scatter=x_hours, y_scatter=y, scatter_label=chlabel, scatter_color="blue",
        x_line=df_TL_MA["Hours"], y_line=df_TL_MA[channelnumber],
        line_label=f"trend line ({Detrending_Method})", line_color="-r",
        x_lim_min=x_scale_min - 6, x_lim_max=x_scale_max, y_lim_tuple=(0, y_scale_max),
        xticks_list=xtickslist, mit_on=miton, mit_val=mit, chlabel=chlabel,
    )

fig.text(0.50, 0.06, "Time (hours)", horizontalalignment="center", fontsize=10)
fig.text(0.08, 0.50, "Bioluminescence", horizontalalignment="center", verticalalignment="center",
         rotation="vertical", fontsize=10)
pp.savefig(fig)
plt.show()
plt.close(fig)

# --- Overall detrended data ---
print("\nPlotting...")
_set_overview_plot_style()
fig = plt.figure(figsize=(11, 8.5))
fig.subplots_adjust(hspace=0.15)
fig.suptitle(f"{Experiment_number} - detrended data ({Detrending_Method})", fontsize=12)

for subplotnumber, k in enumerate(chlist, start=1):
    channelnumber = str(k).zfill(2)
    ax = plt.subplot(subp_raw, subp_col, subplotnumber)
    chlabel = "Ch # " + channelnumber + " - detrend"

    # No y_lim_tuple here: let detrended data auto-scale, as in the original.
    plot_channel_subplot(
        ax=ax,
        x_scatter=dtdf["Hours"], y_scatter=dtdf[channelnumber], scatter_label=chlabel, scatter_color="blue",
        x_line=dtdf_5PMA["Hours"], y_line=dtdf_5PMA[channelnumber], line_label="", line_color="-r",
        x_lim_min=x_scale_min - 6, x_lim_max=x_scale_max, y_lim_tuple=None,
        xticks_list=xtickslist, mit_on=miton, mit_val=mit, chlabel=chlabel,
    )

fig.text(0.50, 0.06, "Time (hours)", horizontalalignment="center", fontsize=10)
fig.text(0.08, 0.50, "Detrended Bioluminescence", horizontalalignment="center", verticalalignment="center",
         rotation="vertical", fontsize=10)
pp.savefig(fig)
plt.show()
plt.close(fig)

# ## Plot per-channel summaries (raw, detrended, actogram)

print("\nPlotting individual channels with actograms...")

plt.rc("font", size=10)
plt.rc("axes", titlesize=10)
plt.rc("axes", labelsize=10)
plt.rc("xtick", labelsize=9)
plt.rc("ytick", labelsize=9)
plt.rc("legend", fontsize=6)
plt.rc("figure", titlesize=12)

for k in chlist:
    channelnumber = str(k).zfill(2)
    plot_individual_channel_summary(
        channelnumber=channelnumber, df=df, df_TL_MA=df_TL_MA, dtdf=dtdf, dtdf_9PMA=dtdf_9PMA,
        Detrending_Method=Detrending_Method,
        Sinc_Filter_Cutoff_Period_Hours=Sinc_Filter_Cutoff_Period_Hours, filter_order=filter_order,
        x_scale_min=x_scale_min, x_scale_max=x_scale_max, xtickslist=xtickslist, miton=miton, mit=mit,
        sub2label=sub2label, sub3label=sub3label, lod=lod, last_time_point=last_time_point,
        time_interval=time_interval,
    )

# ## Fit a damped sine curve to each channel

print("\nPerforming Damped Sine Curve Fitting...")

dtdf_filtered = dtdf[(dtdf["Hours"] >= Start_Hour_for_Fitting) & (dtdf["Hours"] <= End_Hour_for_Fitting)].copy()
all_fit_results = []

plt.rc("font", size=8)
plt.rc("axes", titlesize=8)
plt.rc("axes", labelsize=8)
plt.rc("xtick", labelsize=7)
plt.rc("ytick", labelsize=7)
plt.rc("legend", fontsize=6)


def _nan_fit_result(channelnumber):
    return {
        "Channel": channelnumber,
        "Fitted Amplitude": np.nan,
        "Fitted Period (Hours)": np.nan,
        "Fitted Phase (radians)": np.nan,
        "Fitted Decay Rate": np.nan,
        "Fitted Offset": np.nan,
    }


for k in chlist:
    channelnumber = str(k).zfill(2)

    signal_data_raw = dtdf_filtered[channelnumber].dropna()
    time_data_raw = dtdf_filtered.loc[signal_data_raw.index, "Hours"]

    if len(signal_data_raw) < 5:  # need at least 5 points for 5 parameters
        print(f"Skipping damped sine fit for Channel {channelnumber} due to insufficient data points "
              f"after filtering (need at least 5, got {len(signal_data_raw)}).")
        all_fit_results.append(_nan_fit_result(channelnumber))
        continue

    # Initial parameter guesses (p0) -- crucial for convergence
    amplitude_guess = (signal_data_raw.max() - signal_data_raw.min()) / 2.0
    if amplitude_guess <= 0:
        amplitude_guess = signal_data_raw.std() * 2
    if amplitude_guess == 0:
        amplitude_guess = 1.0

    period_guess = 24.0  # default to a circadian (24h) rhythm
    phase_guess = 0.0
    decay_rate_guess = 0.01
    offset_guess = signal_data_raw.mean()

    lower_bounds = [0, 12.0, -2 * np.pi, 0, -np.inf]
    upper_bounds = [np.inf, 60.0, 2 * np.pi, 0.5, np.inf]

    try:
        time_for_model = time_data_raw - time_data_raw.min()
        params, _covariance = curve_fit(
            damped_sine_model, time_for_model, signal_data_raw,
            p0=[amplitude_guess, period_guess, phase_guess, decay_rate_guess, offset_guess],
            bounds=(lower_bounds, upper_bounds), maxfev=10000,
        )
        fitted_amplitude, fitted_period, fitted_phase, fitted_decay_rate, fitted_offset = params
        normalized_phase = fitted_phase % (2 * np.pi)

        all_fit_results.append({
            "Channel": channelnumber,
            "Fitted Amplitude": fitted_amplitude,
            "Fitted Period (Hours)": fitted_period,
            "Fitted Phase (radians)": normalized_phase,
            "Fitted Decay Rate": fitted_decay_rate,
            "Fitted Offset": fitted_offset,
        })

        # --- Plot the fit (raw data on top, detrended data + fit below) ---
        fig, (ax_raw, ax_detrended) = plt.subplots(2, 1, figsize=(8.5, 11), sharex=True)
        fig.suptitle(f"{Experiment_number} - Damped Sine Fit Channel {channelnumber}", fontsize=12)

        raw_fit_curve = damped_sine_model(time_for_model, *params) + df_TL_MA.loc[time_data_raw.index, channelnumber]

        ax_raw.plot(df["Hours"], df[channelnumber], "o", markersize=2, color="gray", label="Full Raw Data")
        ax_raw.plot(df_TL_MA["Hours"], df_TL_MA[channelnumber], "k--", linewidth=1.0, label="Trend Line")
        ax_raw.plot(time_data_raw, raw_fit_curve, "r-", linewidth=1.5, label="Raw Data Fit")
        ax_raw.set_ylabel("Bioluminescence", fontsize=10)
        ax_raw.set_title(f"Raw Data Fit (P={fitted_period:.2f}h A={fitted_amplitude:.2f} Ph={normalized_phase:.2f})", fontsize=11)
        ax_raw.legend(loc="upper right", fontsize=8)
        ax_raw.grid(True, linestyle="--", alpha=0.7)
        ax_raw.set_ylim(bottom=0)

        ax_detrended.plot(dtdf["Hours"], dtdf[channelnumber], "o", markersize=2, color="gray", label="Full Detrended Data")
        ax_detrended.plot(time_data_raw, signal_data_raw, "o", markersize=2, color="blue",
                           label=f"Data for Fitting (from {Start_Hour_for_Fitting}h to {End_Hour_for_Fitting}h)")
        ax_detrended.plot(time_data_raw, damped_sine_model(time_for_model, *params), "r-", linewidth=1.5, label="Damped Sine Fit")
        ax_detrended.set_xlabel("Time (Hours)", fontsize=10)
        ax_detrended.set_ylabel("Detrended Bioluminescence", fontsize=10)
        ax_detrended.set_title(f"Detrended Data Fit (Dec={fitted_decay_rate:.6f} Off={fitted_offset:.6f})", fontsize=11)
        ax_detrended.legend(loc="upper right", fontsize=8)
        ax_detrended.grid(True, linestyle="--", alpha=0.7)

        ax_detrended.set_xlim(x_scale_min, x_scale_max)
        ax_detrended.xaxis.set_major_locator(MultipleLocator(base=24, offset=0))
        ax_detrended.xaxis.set_minor_locator(MultipleLocator(base=12, offset=0))

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        pp.savefig(fig)
        plt.show()
        plt.close(fig)

    except (RuntimeError, ValueError) as e:
        print(f"Could not fit damped sine to Channel {channelnumber}: {e}")
        all_fit_results.append(_nan_fit_result(channelnumber))

pp.close()

df_fit_results = pd.DataFrame(all_fit_results)
print("\nDamped Sine Curve Fitting Results:")
display(df_fit_results)

new_fitting_excel_filename = f"{Experiment_number}_damped_sine_fit_results.xlsx"
with pd.ExcelWriter(new_fitting_excel_filename) as writer:
    df_fit_results.to_excel(writer, sheet_name="Damped Sine Fit", index=False)
print(f"\nDamped sine fitting results added to new Excel file: {new_fitting_excel_filename}")

# ## Download the results

files.download(outputexcelfilename)
files.download(outputexcelfilename2)
files.download(plot_output_pdf)
files.download(zip_output_filename)
files.download(new_fitting_excel_filename)

endtime = datetime.now(timezone.utc)
elapsed = endtime - sttime
print(f"\nElapsed time: {elapsed.seconds} seconds")
print(f"Completed at {endtime.strftime('%Y-%m-%d %H:%M:%S')} (UTC)\n")
